In [1]:
import numpy as np
import pandas as pd
import re

In [2]:
#import genotype data
genotype = pd.read_csv(
    "../data/raw/Pf8_drug_resistance_marker_genotypes.tsv",
    sep="\t"
)

In [3]:
# viewing data
genotype

,Sample,crt_72[C],crt_74[M],crt_75[N],crt_76[K],crt_72-76[CVMNK],crt_93[T],crt_97[H],crt_218[I],crt_220[A],...,mdr1_1034[S],mdr1_1042[N],mdr1_1226[F],mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T],kelch13_349-726_ns_changes,mdr1_dup_call,pm2_dup_call
0,FP0008-C,C,"I,M","E,N","T,K","CVIET,CVMNK",T,H,I,"S,A",...,S,N,F,D,VD,D,T,NaN,0,0
1,FP0009-C,C,I,E,T,CVIET,T,H,I,S,...,S,N,F,Y,VD,D,T,NaN,0,0
2,FP0010-CW,C,"I,M","E,N","T,K","CVIET,CVMNK",T,H,I,"S,A",...,S,N,F,D,VD,D,T,NaN,0,0
3,FP0011-CW,C,"I,M","E,N","T,K","CVIET,CVMNK",T,H,I,"S,A",...,S,N,F,D,VD,D,T,NaN,0,0
4,FP0012-CW,C,I,E,T,CVIET,T,H,I,S,...,S,N,F,D,VD,D,T,NaN,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24404,SPT92049,C,M,N,K,CVMNK,T,H,I,A,...,S,N,F,-,VD,D,T,-,-1,-1
24405,SPT92054,C,M,N,K,CVMNK,T,H,I,A,...,S,N,F,D,VD,D,T,NaN,0,-1
24406,SPT92057,C,M,N,K,CVMNK,T,H,I,A,...,S,N,F,D,VD,D,T,-,-1,-1
24407,SPT94772,C,I,E,T,CVIET,T,H,I,S,...,-,-,-,-,VD,D,-,-,-1,-1


### Data exploration

In [4]:
print(genotype["crt_76[K]"].value_counts(dropna=False))

crt_76[K]
T       12432
K       10768
T,K       607
K,T       550
-          48
T,Q         3
T,Q*        1
Name: count, dtype: int64


In [5]:
# viewing all column count data
genotype_dic = {}
for col in genotype.columns:
    value_count = genotype[col].value_counts(dropna = False)
    genotype_dic[col] = value_count

In [6]:
print(genotype_dic)

{'Sample': Sample
FP0008-C     1
FP0009-C     1
FP0010-CW    1
FP0011-CW    1
FP0012-CW    1
            ..
SPT92049     1
SPT92054     1
SPT92057     1
SPT94772     1
SPT94773     1
Name: count, Length: 24409, dtype: int64, 'crt_72[C]': crt_72[C]
C      23955
S        402
-         34
S,C        6
C,S        5
C,Y        5
Y          2
Name: count, dtype: int64, 'crt_74[M]': crt_74[M]
I      11883
M      11318
I,M      612
M,I      544
-         52
Name: count, dtype: int64, 'crt_75[N]': crt_75[N]
N       11205
E       11017
D         839
E,N       594
N,E       525
E,D        71
D,E        62
-          52
N,D        19
D,N         9
E,K         7
N,E*        3
E,N*        3
K           2
K,E         1
Name: count, dtype: int64, 'crt_76[K]': crt_76[K]
T       12432
K       10768
T,K       607
K,T       550
-          48
T,Q         3
T,Q*        1
Name: count, dtype: int64, 'crt_72-76[CVMNK]': crt_72-76[CVMNK]
CVIET           10898
CVMNK           10749
CVIDT             839
CVIET,CV

In [7]:
genotype.columns

Index(['Sample', 'crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_76[K]',
       'crt_72-76[CVMNK]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]',
       'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]',
       'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_51[N]', 'dhfr_59[C]',
       'dhfr_108[S]', 'dhfr_164[I]', 'dhfr_306[S]', 'dhps_436[S]',
       'dhps_437[G]', 'dhps_540[K]', 'dhps_581[A]', 'dhps_613[A]',
       'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]',
       'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]',
       'fd_193[D]', 'mdr2_484[T]', 'kelch13_349-726_ns_changes',
       'mdr1_dup_call', 'pm2_dup_call'],
      dtype='str')

In [8]:
# handling kelchi data
genotype['kelch13_349-726_ns_changes'].unique()

<StringArray>
[    nan,     '-', 'a676s', 'p441s', 'E612D', 'A676S', 'S522C', 'a578s',
 's522i', 'e509d',
 ...
 'v566i', 'p527s', 'a427v', 'v566l', 'd641n', 'S364Y', 'V487E', 's477y',
 'T350S', 'c469y']
Length: 266, dtype: str

## Classifying known biomarkers

In [9]:
genotype.columns

Index(['Sample', 'crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_76[K]',
       'crt_72-76[CVMNK]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]',
       'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]',
       'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_51[N]', 'dhfr_59[C]',
       'dhfr_108[S]', 'dhfr_164[I]', 'dhfr_306[S]', 'dhps_436[S]',
       'dhps_437[G]', 'dhps_540[K]', 'dhps_581[A]', 'dhps_613[A]',
       'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]',
       'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]',
       'fd_193[D]', 'mdr2_484[T]', 'kelch13_349-726_ns_changes',
       'mdr1_dup_call', 'pm2_dup_call'],
      dtype='str')

In [10]:
# creating a dataframe for known resistance biomakers
encoded = pd.DataFrame()

encoded["Sample"] = genotype["Sample"]

In [11]:
# markers associated with drug resistance

associated_marker = {

    "dhfr_51[N]": {
        "gene": "dhfr",
        "wildtype": "N",
        "resistant": {"I"}
    },

    "dhfr_59[C]": {
        "gene": "dhfr",
        "wildtype": "C",
        "resistant": {"R"}
    },

    "dhfr_164[I]": {
        "gene": "dhfr",
        "wildtype": "I",
        "resistant": {"L"}
    },


    "dhps_540[K]": {
        "gene": "dhps",
        "wildtype": "K",
        "resistant": {"E"}
    },

    "dhps_581[A]": {
        "gene": "dhps",
        "wildtype": "A",
        "resistant": {"G"}
    },

    "dhps_613[A]": {
        "gene": "dhps",
        "wildtype": "A",
        "resistant": {"S", "T"}
    }
}

In [12]:
marker_info = {
    "crt_76[K]": {
        "wildtype": "K",
        "resistant": {"T"}
    },

    "dhfr_108[S]": {
        "wildtype": "S",
        "resistant": {"N"}
    },

    "dhps_437[G]": {
        "wildtype": "A",
        "resistant": {"G"}
    }
}

In [13]:
# known columns
known_marker = {**marker_info,**associated_marker }
known_encoder = []

for col, item in known_marker.items():
    known_encoder.append(col)


In [14]:
def classify_known_marker(value, wildtype, resistance_alleles):

    if pd.isna(value):
        return "Missing"

    value = str(value).strip()

    if value in {".", "-", "*", "!"}:
        return "Missing"

    if "," in value:

        haps = [x.strip().upper() for x in value.split(",")]

        if any(x in {"", "-", "*", "!"} for x in haps):
            return "Missing"

        if any(x in resistance_alleles for x in haps):
            return "Resistant"

        if all(x == wildtype.upper() for x in haps):
            return "Sensitive"

        return "Missing"

    allele = value.upper()

    if allele in resistance_alleles:
        return "Resistant"

    if allele == wildtype.upper():
        return "Sensitive"

    return "Missing"

In [15]:
# handle mutations causing drug resistance
for column, info in marker_info.items():

    encoded[column] = genotype[column].apply(
        lambda x: classify_known_marker(
            x,
            wildtype=info["wildtype"],
            resistance_alleles=info["resistant"]
        )
    )

In [16]:
# handle mutations associated with resistance
for column, info in associated_marker.items():
    new_column = f"{column}_assoc"
    encoded[new_column] = genotype[column].apply(
        lambda x: classify_known_marker(
            x,
            wildtype=info["wildtype"],
            resistance_alleles=info["resistant"]
        )
    )

In [17]:
encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive


#### Handling kelch13

In [18]:
kelch13_col = "kelch13_349-726_ns_changes"

In [19]:
k13_resistance = {
    "F446I",
    "N458Y",
    "C469Y",
    "M476I",
    "Y493H",
    "R539T",
    "I543T",
    "P553L",
    "R561H",
    "P574L",
    "C580Y",
    "R622I",
    "A675V"
}

In [20]:
# classifying k13
def classify_k13(value):

    if pd.isna(value):
        return "Missing"

    value = str(value).strip()

    if value in {"", ".", "-", "*", "!"}:
        return "Missing"

    # Some cells may contain multiple mutations
    mutations = [
        x.strip().upper()
        for x in value.split(",")
    ]

    if any(m in k13_resistance for m in mutations):
        return "Resistant"

    return "Other"

In [21]:
encoded["kelch13_known"] = genotype[
    "kelch13_349-726_ns_changes"
].apply(classify_k13)

In [22]:
encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,kelch13_known
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Missing
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing


#### Handling resistance due to multiple column copy

In [23]:
# encoding if multiple copies associated with drug resistance
def copy_encoder(value):

    # Missing
    if pd.isna(value) or str(value).strip() == "":
        return "Missing"

    value = str(value).strip()

    # ---------- Special: copy number ----------
    # ----- Special: copy number ----------
    
    if int(value) == 0:
        return "Sensitive"
    elif int(value) == 1:
        return "Resistant"
    elif value in {"-", "*", "!"}:
        return "missing"
    else:

        return "rare"


In [24]:
encoded[["mdr1_dup_call", "pm2_dup_call"]] = genotype[["mdr1_dup_call", "pm2_dup_call"]].map(copy_encoder)


In [25]:
encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,kelch13_known,mdr1_dup_call,pm2_dup_call
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive


In [26]:
encoded.to_csv('Important_genotype.csv',index = True)

## Handling other columns

In [27]:
genotype_col = genotype.columns.to_list()
#known_marker.append(["mdr1_dup_call", "pm2_dup_call","kelch13_349-726_ns_changes"])
len(genotype_col)
genotype_col


['Sample',
 'crt_72[C]',
 'crt_74[M]',
 'crt_75[N]',
 'crt_76[K]',
 'crt_72-76[CVMNK]',
 'crt_93[T]',
 'crt_97[H]',
 'crt_218[I]',
 'crt_220[A]',
 'crt_271[Q]',
 'crt_326[N]',
 'crt_333[T]',
 'crt_353[G]',
 'crt_356[I]',
 'crt_371[R]',
 'dhfr_16[N]',
 'dhfr_51[N]',
 'dhfr_59[C]',
 'dhfr_108[S]',
 'dhfr_164[I]',
 'dhfr_306[S]',
 'dhps_436[S]',
 'dhps_437[G]',
 'dhps_540[K]',
 'dhps_581[A]',
 'dhps_613[A]',
 'exo_415[E]',
 'mdr1_86[N]',
 'mdr1_184[Y]',
 'mdr1_1034[S]',
 'mdr1_1042[N]',
 'mdr1_1226[F]',
 'mdr1_1246[D]',
 'arps10_127-128[VD]',
 'fd_193[D]',
 'mdr2_484[T]',
 'kelch13_349-726_ns_changes',
 'mdr1_dup_call',
 'pm2_dup_call']

In [28]:
rare_col = [
    col for col in genotype.columns
    if col not in known_marker
    and col != "Sample"
]

In [29]:
exclude = {
    "Sample",
    "crt_72-76[CVMNK]",
    "kelch13_349-726_ns_changes",
    "mdr1_dup_call",
    "pm2_dup_call"
}

rare_col = [
    col for col in genotype.columns
    if col not in known_marker
    and col not in exclude
]

print(rare_col)

['crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]', 'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]', 'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_306[S]', 'dhps_436[S]', 'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]', 'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]', 'fd_193[D]', 'mdr2_484[T]']


### Handling rare mutations

In [30]:
# extracting rfference ammino acide for rare columns
import re

rare_ref = {}

for col in rare_col:
    match = re.search(r'\[([A-Z]+)\]', col)
    
    if match:
        rare_ref[col] = match.group(1)

print(rare_ref)

{'crt_72[C]': 'C', 'crt_74[M]': 'M', 'crt_75[N]': 'N', 'crt_93[T]': 'T', 'crt_97[H]': 'H', 'crt_218[I]': 'I', 'crt_220[A]': 'A', 'crt_271[Q]': 'Q', 'crt_326[N]': 'N', 'crt_333[T]': 'T', 'crt_353[G]': 'G', 'crt_356[I]': 'I', 'crt_371[R]': 'R', 'dhfr_16[N]': 'N', 'dhfr_306[S]': 'S', 'dhps_436[S]': 'S', 'exo_415[E]': 'E', 'mdr1_86[N]': 'N', 'mdr1_184[Y]': 'Y', 'mdr1_1034[S]': 'S', 'mdr1_1042[N]': 'N', 'mdr1_1226[F]': 'F', 'mdr1_1246[D]': 'D', 'arps10_127-128[VD]': 'VD', 'fd_193[D]': 'D', 'mdr2_484[T]': 'T'}


In [31]:
# encoding rare (undetermined column)
def encode_rare(value, ref):
    
    # Missing value
    if pd.isna(value):
        return "Missing"
    
    value = str(value).strip()
    
    # Missing/uncertain symbols
    if value == "" or any(x in value for x in ["*", "!", "."]):
        return "Missing"
    
    # Split heterozygous calls
    alleles = [x.strip() for x in value.split(",")]
    
    # If ALL observed alleles are reference
    if all(allele == ref for allele in alleles):
        return "sensitive"
    
    # Any non-reference mutation
    return "Rare"

In [32]:


for col in rare_col:
    
    ref = rare_ref[col]
    
    encoded[col] = genotype[col].apply(
        lambda x: encode_rare(x, ref)
    )

In [33]:
encoded

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,...,exo_415[E],mdr1_86[N],mdr1_184[Y],mdr1_1034[S],mdr1_1042[N],mdr1_1226[F],mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T]
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,Rare,Rare,sensitive,sensitive,sensitive,Rare,sensitive,sensitive,sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,Rare,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,Rare,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24404,SPT92049,Sensitive,Resistant,Missing,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,Rare,sensitive,sensitive,sensitive
24405,SPT92054,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
24406,SPT92057,Sensitive,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive
24407,SPT94772,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,Rare,Rare,Rare,Rare,Rare,sensitive,sensitive,Rare


In [34]:
encoded.shape

(24409, 39)

# Handling the drug phenotype

In [35]:
# loading sample metadata
metadata = pd.read_csv("../data/raw/Pf8-samples.csv")

In [36]:
metadata.columns

Index(['sample_id', 'year', 'qc_pass', 'study_id', 'region', 'country',
       'country_id', 'site', 'site_id', 'ARTresistant', 'CQresistant',
       'MQresistant', 'PPQresistant', 'PYRresistant', 'SDXresistant'],
      dtype='str')

In [37]:
# dropping unnecessary columns
del_col = ['year', 'qc_pass', 'study_id', 'region', 'country', 'country_id', 'site', 'site_id']

In [38]:
merged_encoded = encoded.merge(metadata, left_on = 'Sample', right_on = 'sample_id', how = 'left')

In [39]:
merged_encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,...,country,country_id,site,site_id,ARTresistant,CQresistant,MQresistant,PPQresistant,PYRresistant,SDXresistant
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,sensitive,sensitive,undetermined,undetermined
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,sensitive,sensitive,resistant,sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,resistant
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,undetermined
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,undetermined,undetermined,resistant,sensitive


In [40]:
merged_encoded.shape

(24409, 54)

In [41]:
# removing unnecessary columns 
merged_encoded.drop(columns = del_col, inplace = True)

In [42]:
merged_encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,...,arps10_127-128[VD],fd_193[D],mdr2_484[T],sample_id,ARTresistant,CQresistant,MQresistant,PPQresistant,PYRresistant,SDXresistant
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,FP0008-C,sensitive,undetermined,sensitive,sensitive,undetermined,undetermined
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,FP0009-C,sensitive,resistant,sensitive,sensitive,resistant,sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,FP0010-CW,sensitive,undetermined,undetermined,undetermined,resistant,resistant
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,FP0011-CW,sensitive,undetermined,undetermined,undetermined,resistant,undetermined
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,FP0012-CW,sensitive,resistant,undetermined,undetermined,resistant,sensitive


In [43]:
merged_encoded['dhfr_51[N]_assoc'].value_counts(dropna=False)

dhfr_51[N]_assoc
Resistant    20083
Sensitive     4029
Missing        297
Name: count, dtype: int64

In [44]:
merged_encoded.info()

<class 'pandas.DataFrame'>
RangeIndex: 24409 entries, 0 to 24408
Data columns (total 46 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Sample              24409 non-null  str  
 1   crt_76[K]           24409 non-null  str  
 2   dhfr_108[S]         24409 non-null  str  
 3   dhps_437[G]         24409 non-null  str  
 4   dhfr_51[N]_assoc    24409 non-null  str  
 5   dhfr_59[C]_assoc    24409 non-null  str  
 6   dhfr_164[I]_assoc   24409 non-null  str  
 7   dhps_540[K]_assoc   24409 non-null  str  
 8   dhps_581[A]_assoc   24409 non-null  str  
 9   dhps_613[A]_assoc   24409 non-null  str  
 10  kelch13_known       24409 non-null  str  
 11  mdr1_dup_call       24409 non-null  str  
 12  pm2_dup_call        24409 non-null  str  
 13  crt_72[C]           24409 non-null  str  
 14  crt_74[M]           24409 non-null  str  
 15  crt_75[N]           24409 non-null  str  
 16  crt_93[T]           24409 non-null  str  
 17  crt_

## Handling target columns

In [45]:
target_df = merged_encoded[['ARTresistant', 'CQresistant',
       'MQresistant', 'PPQresistant', 'PYRresistant', 'SDXresistant']]

In [46]:
target_col = ['ARTresistant', 'CQresistant','MQresistant', 'PPQresistant', 'PYRresistant', 'SDXresistant']

In [47]:
pd.set_option('display.max_rows', None)
for col in target_df.columns:
    print(f"\n{col}")
    print(target_df[col].value_counts(dropna=False))


ARTresistant
ARTresistant
sensitive       17147
resistant        3905
undetermined     3357
Name: count, dtype: int64

CQresistant
CQresistant
resistant       12432
sensitive       10768
undetermined     1209
Name: count, dtype: int64

MQresistant
MQresistant
sensitive       12598
undetermined    11153
resistant         658
Name: count, dtype: int64

PPQresistant
PPQresistant
sensitive       13467
undetermined    10454
resistant         488
Name: count, dtype: int64

PYRresistant
PYRresistant
resistant       21964
undetermined     1230
sensitive        1215
Name: count, dtype: int64

SDXresistant
SDXresistant
resistant       19270
sensitive        3601
undetermined     1538
Name: count, dtype: int64


### Encoding the y_target column

In [48]:
import numpy as np

Y = merged_encoded[target_col].replace({
    "sensitive": 0,
    "resistant": 1,
    "undetermined": 0
})

In [49]:
Y.head()

,ARTresistant,CQresistant,MQresistant,PPQresistant,PYRresistant,SDXresistant
0,0,0,0,0,0,0
1,0,1,0,0,1,0
2,0,0,0,0,1,1
3,0,0,0,0,1,0
4,0,1,0,0,1,0


## Encoding features variable

In [50]:
features_df = merged_encoded.drop(columns = target_col)

In [51]:
features_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 24409 entries, 0 to 24408
Data columns (total 40 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Sample              24409 non-null  str  
 1   crt_76[K]           24409 non-null  str  
 2   dhfr_108[S]         24409 non-null  str  
 3   dhps_437[G]         24409 non-null  str  
 4   dhfr_51[N]_assoc    24409 non-null  str  
 5   dhfr_59[C]_assoc    24409 non-null  str  
 6   dhfr_164[I]_assoc   24409 non-null  str  
 7   dhps_540[K]_assoc   24409 non-null  str  
 8   dhps_581[A]_assoc   24409 non-null  str  
 9   dhps_613[A]_assoc   24409 non-null  str  
 10  kelch13_known       24409 non-null  str  
 11  mdr1_dup_call       24409 non-null  str  
 12  pm2_dup_call        24409 non-null  str  
 13  crt_72[C]           24409 non-null  str  
 14  crt_74[M]           24409 non-null  str  
 15  crt_75[N]           24409 non-null  str  
 16  crt_93[T]           24409 non-null  str  
 17  crt_

In [52]:
features_df.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,...,mdr1_86[N],mdr1_184[Y],mdr1_1034[S],mdr1_1042[N],mdr1_1226[F],mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T],sample_id
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,FP0008-C
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,Rare,sensitive,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,FP0009-C
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,FP0010-CW
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,FP0011-CW
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,FP0012-CW


In [53]:
features_df = features_df.drop(
    columns=["Sample", "sample_id"],
    errors="ignore"
).copy()

In [54]:
for col in features_df.columns:
    print(f"\n===== {col} =====")
    print(features_df[col].value_counts(dropna=False))


===== crt_76[K] =====
crt_76[K]
Resistant    13593
Sensitive    10768
Missing         48
Name: count, dtype: int64

===== dhfr_108[S] =====
dhfr_108[S]
Resistant    22780
Sensitive     1215
Missing        414
Name: count, dtype: int64

===== dhps_437[G] =====
dhps_437[G]
Resistant    20538
Sensitive     3601
Missing        270
Name: count, dtype: int64

===== dhfr_51[N]_assoc =====
dhfr_51[N]_assoc
Resistant    20083
Sensitive     4029
Missing        297
Name: count, dtype: int64

===== dhfr_59[C]_assoc =====
dhfr_59[C]_assoc
Resistant    22051
Sensitive     2034
Missing        324
Name: count, dtype: int64

===== dhfr_164[I]_assoc =====
dhfr_164[I]_assoc
Sensitive    19106
Resistant     5032
Missing        271
Name: count, dtype: int64

===== dhps_540[K]_assoc =====
dhps_540[K]_assoc
Sensitive    12850
Resistant     8386
Missing       3173
Name: count, dtype: int64

===== dhps_581[A]_assoc =====
dhps_581[A]_assoc
Sensitive    18695
Resistant     5349
Missing        365
Name: count, d

In [55]:
features_df = features_df[
    features_df['dhps_613[A]_assoc'] != "Other"
].copy()

In [56]:
for col in features_df.columns:
    print(f"\n===== {col} =====")
    print(features_df[col].value_counts(dropna=False))


===== crt_76[K] =====
crt_76[K]
Resistant    13593
Sensitive    10768
Missing         48
Name: count, dtype: int64

===== dhfr_108[S] =====
dhfr_108[S]
Resistant    22780
Sensitive     1215
Missing        414
Name: count, dtype: int64

===== dhps_437[G] =====
dhps_437[G]
Resistant    20538
Sensitive     3601
Missing        270
Name: count, dtype: int64

===== dhfr_51[N]_assoc =====
dhfr_51[N]_assoc
Resistant    20083
Sensitive     4029
Missing        297
Name: count, dtype: int64

===== dhfr_59[C]_assoc =====
dhfr_59[C]_assoc
Resistant    22051
Sensitive     2034
Missing        324
Name: count, dtype: int64

===== dhfr_164[I]_assoc =====
dhfr_164[I]_assoc
Sensitive    19106
Resistant     5032
Missing        271
Name: count, dtype: int64

===== dhps_540[K]_assoc =====
dhps_540[K]_assoc
Sensitive    12850
Resistant     8386
Missing       3173
Name: count, dtype: int64

===== dhps_581[A]_assoc =====
dhps_581[A]_assoc
Sensitive    18695
Resistant     5349
Missing        365
Name: count, d

In [57]:
features_df['crt_371[R]'].value_counts()

crt_371[R]
Rare         12455
sensitive    11954
Name: count, dtype: int64

### Encoding the X features

In [58]:
# defining different columns 
known_cols = list(marker_info.keys())
known_cols.append("mdr1_dup_call")
known_cols.append("pm2_dup_call")


associated_cols = list(associated_marker.keys())

rare_cols = rare_col

In [59]:
print("Known:", known_cols)
print("Associated:", associated_cols)
print("Rare:", rare_cols)

Known: ['crt_76[K]', 'dhfr_108[S]', 'dhps_437[G]', 'mdr1_dup_call', 'pm2_dup_call']
Associated: ['dhfr_51[N]', 'dhfr_59[C]', 'dhfr_164[I]', 'dhps_540[K]', 'dhps_581[A]', 'dhps_613[A]']
Rare: ['crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]', 'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]', 'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_306[S]', 'dhps_436[S]', 'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]', 'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]', 'fd_193[D]', 'mdr2_484[T]']


In [60]:
print([c for c in features_df.columns if 'dhfr' in c or 'dhps' in c])

['dhfr_108[S]', 'dhps_437[G]', 'dhfr_51[N]_assoc', 'dhfr_59[C]_assoc', 'dhfr_164[I]_assoc', 'dhps_540[K]_assoc', 'dhps_581[A]_assoc', 'dhps_613[A]_assoc', 'dhfr_16[N]', 'dhfr_306[S]', 'dhps_436[S]']


In [61]:
assoc_col = [c for c in encoded.columns if '_assoc' in c]

In [62]:
# encoding known and associated columns

X_known = features_df[known_cols].replace({
    "Resistant": 1,
    "Sensitive": 0,
    "sensitive": 0,
    "Missing": 0,
    "rare": 0
})

X_associated = features_df[assoc_col].replace({
    "Resistant": 1,
    "Sensitive": 0,
    "sensitive": 0,
    "Missing": 0,
    "rare": 0
})

In [63]:
# viewing columns associated with known resistance
for col in known_cols:
    print(col, X_known[col].value_counts(dropna=False))

crt_76[K] crt_76[K]
1    13593
0    10816
Name: count, dtype: int64
dhfr_108[S] dhfr_108[S]
1    22780
0     1629
Name: count, dtype: int64
dhps_437[G] dhps_437[G]
1    20538
0     3871
Name: count, dtype: int64
mdr1_dup_call mdr1_dup_call
0    23707
1      702
Name: count, dtype: int64
pm2_dup_call pm2_dup_call
0    22329
1     2080
Name: count, dtype: int64


In [64]:
# viewing columns assocciated with resistance associations
for col in assoc_col:
    print(col, X_associated[col].value_counts(dropna=False))

dhfr_51[N]_assoc dhfr_51[N]_assoc
1    20083
0     4326
Name: count, dtype: int64
dhfr_59[C]_assoc dhfr_59[C]_assoc
1    22051
0     2358
Name: count, dtype: int64
dhfr_164[I]_assoc dhfr_164[I]_assoc
0    19377
1     5032
Name: count, dtype: int64
dhps_540[K]_assoc dhps_540[K]_assoc
0    16023
1     8386
Name: count, dtype: int64
dhps_581[A]_assoc dhps_581[A]_assoc
0    19060
1     5349
Name: count, dtype: int64
dhps_613[A]_assoc dhps_613[A]_assoc
0    23050
1     1359
Name: count, dtype: int64


In [65]:
# Handling rare mutations
X_rare = features_df[rare_cols].replace({
    "sensitive": 0,
    "Sensitive": 0,
    "Rare": 0,
    "Missing": 0
})

In [66]:
# merging 
X = pd.concat(
    [X_associated, X_rare],
    axis=1
)

print("X shape:", X.shape)
X.head()

X shape: (24409, 32)


,dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,crt_72[C],crt_74[M],crt_75[N],crt_93[T],...,exo_415[E],mdr1_86[N],mdr1_184[Y],mdr1_1034[S],mdr1_1042[N],mdr1_1226[F],mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T]
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,1,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [67]:
X.shape

(24409, 32)

In [68]:
Y.shape

(24409, 6)

In [69]:
y = Y.apply(pd.to_numeric, errors="coerce")

In [70]:
X = X.apply(pd.to_numeric, errors="coerce")

## Model

In [71]:
mask = Y.notna().all(axis=1)

X_model = X.loc[mask]
y_model = Y.loc[mask]

print(X_model.shape)
print(y_model.shape)

(24409, 32)
(24409, 6)


In [72]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.2,
    random_state=42
)

In [73]:
mask = y.notna().all(axis=1)

X_model = X.loc[mask].copy()
y_model = y.loc[mask].copy()

In [74]:
print("X:", X_model.shape)
print("y:", y_model.shape)

print(y_model.dtypes)
print(y_model.apply(lambda x: x.unique()))

X: (24409, 32)
y: (24409, 6)
ARTresistant    int64
CQresistant     int64
MQresistant     int64
PPQresistant    int64
PYRresistant    int64
SDXresistant    int64
dtype: object
   ARTresistant  CQresistant  MQresistant  PPQresistant  PYRresistant  \
0             0            0            0             0             0   
1             1            1            1             1             1   

   SDXresistant  
0             0  
1             1  


In [75]:
from sklearn.model_selection import train_test_split
from sklearn.utils.multiclass import type_of_target

# Create a fresh train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# Check the target
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("y_train dtype:")
print(y_train.dtypes)

print("\nTarget type:")
print(type_of_target(y_train))

X_train: (19527, 32)
y_train: (19527, 6)
y_train dtype:
ARTresistant    int64
CQresistant     int64
MQresistant     int64
PPQresistant    int64
PYRresistant    int64
SDXresistant    int64
dtype: object

Target type:
multilabel-indicator


In [76]:
from sklearn.utils.multiclass import type_of_target

print(type_of_target(y_model))

multilabel-indicator


In [77]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total

In [78]:
y_pred = rf.predict(X_test)

In [79]:
print(y_pred.shape)

(4882, 6)


In [80]:
from sklearn.metrics import classification_report

for i, drug in enumerate(y.columns):
    print(f"\n===== {drug} =====")
    print(
        classification_report(
            y_test.iloc[:, i],
            y_pred[:, i]
        )
    )


===== ARTresistant =====
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      4071
           1       0.87      0.67      0.76       811

    accuracy                           0.93      4882
   macro avg       0.90      0.83      0.86      4882
weighted avg       0.93      0.93      0.93      4882


===== CQresistant =====
              precision    recall  f1-score   support

           0       0.68      0.93      0.78      2399
           1       0.89      0.57      0.70      2483

    accuracy                           0.75      4882
   macro avg       0.79      0.75      0.74      4882
weighted avg       0.79      0.75      0.74      4882


===== MQresistant =====
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      4756
           1       0.00      0.00      0.00       126

    accuracy                           0.97      4882
   macro avg       0.49      0.50      0.49      488

/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [81]:
#self
y_self_pred = rf.predict(X_train)

In [82]:
for i, drug in enumerate(y.columns):
    print(f"\n===== {drug} =====")
    print(
        classification_report(
            y_train.iloc[:, i],
            y_self_pred[:, i]
        )
    )


===== ARTresistant =====
              precision    recall  f1-score   support

           0       0.93      0.98      0.95     16433
           1       0.83      0.62      0.71      3094

    accuracy                           0.92     19527
   macro avg       0.88      0.80      0.83     19527
weighted avg       0.92      0.92      0.91     19527


===== CQresistant =====
              precision    recall  f1-score   support

           0       0.67      0.92      0.78      9578
           1       0.89      0.56      0.69      9949

    accuracy                           0.74     19527
   macro avg       0.78      0.74      0.73     19527
weighted avg       0.78      0.74      0.73     19527


===== MQresistant =====
              precision    recall  f1-score   support

           0       0.97      1.00      0.99     18995
           1       0.00      0.00      0.00       532

    accuracy                           0.97     19527
   macro avg       0.49      0.50      0.49     1952

/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

## XGBoost

In [83]:
%pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [84]:
from xgboost import XGBClassifier

In [85]:
X_xgb = X.copy()

X_xgb.columns = (
    X_xgb.columns
    .str.replace("[", "_", regex=False)
    .str.replace("]", "_", regex=False)
    .str.replace("<", "_", regex=False)
)

print(X_xgb.columns.tolist())

['dhfr_51_N__assoc', 'dhfr_59_C__assoc', 'dhfr_164_I__assoc', 'dhps_540_K__assoc', 'dhps_581_A__assoc', 'dhps_613_A__assoc', 'crt_72_C_', 'crt_74_M_', 'crt_75_N_', 'crt_93_T_', 'crt_97_H_', 'crt_218_I_', 'crt_220_A_', 'crt_271_Q_', 'crt_326_N_', 'crt_333_T_', 'crt_353_G_', 'crt_356_I_', 'crt_371_R_', 'dhfr_16_N_', 'dhfr_306_S_', 'dhps_436_S_', 'exo_415_E_', 'mdr1_86_N_', 'mdr1_184_Y_', 'mdr1_1034_S_', 'mdr1_1042_N_', 'mdr1_1226_F_', 'mdr1_1246_D_', 'arps10_127-128_VD_', 'fd_193_D_', 'mdr2_484_T_']


In [86]:
from sklearn.model_selection import train_test_split

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X_xgb,
    y,
    test_size=0.2,
    random_state=42
)

In [87]:
from xgboost import XGBClassifier

xgb_models = {}
y_pred_xgb = {}

for drug in y.columns:

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train_xgb,
        y_train_xgb[drug]
    )

    xgb_models[drug] = model
    y_pred_xgb[drug] = model.predict(X_test_xgb)

In [88]:
from sklearn.metrics import classification_report

for drug in y.columns:

    print(f"\n===== {drug} =====")

    print(
        classification_report(
            y_test[drug],
            y_pred_xgb[drug]
        )
    )


===== ARTresistant =====
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      4071
           1       0.87      0.67      0.76       811

    accuracy                           0.93      4882
   macro avg       0.90      0.83      0.86      4882
weighted avg       0.93      0.93      0.93      4882


===== CQresistant =====
              precision    recall  f1-score   support

           0       0.68      0.93      0.78      2399
           1       0.89      0.57      0.70      2483

    accuracy                           0.75      4882
   macro avg       0.79      0.75      0.74      4882
weighted avg       0.79      0.75      0.74      4882


===== MQresistant =====
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      4756
           1       0.00      0.00      0.00       126

    accuracy                           0.97      4882
   macro avg       0.49      0.50      0.49      488

/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [89]:
from sklearn.metrics import f1_score, precision_score, recall_score

results = []

for drug in y.columns:

    i = y.columns.get_loc(drug)

    results.append({
        "Drug": drug,
        "RF_F1": f1_score(
            y_test[drug],
            y_pred[:, i]
        ),
        "XGB_F1": f1_score(
            y_test[drug],
            y_pred_xgb[drug]
        ),
        "RF_Recall": recall_score(
            y_test[drug],
            y_pred[:, i]
        ),
        "XGB_Recall": recall_score(
            y_test[drug],
            y_pred_xgb[drug]
        )
    })

comparison = pd.DataFrame(results)

print(comparison)

           Drug     RF_F1    XGB_F1  RF_Recall  XGB_Recall
0  ARTresistant  0.760250  0.760250   0.674476    0.674476
1   CQresistant  0.696657  0.696657   0.570681    0.570681
2   MQresistant  0.000000  0.000000   0.000000    0.000000
3  PPQresistant  0.000000  0.000000   0.000000    0.000000
4  PYRresistant  0.971435  0.971435   0.990180    0.990180
5  SDXresistant  0.892617  0.892617   0.947738    0.947738


In [90]:
from sklearn.metrics import classification_report

for drug in y.columns:
    print(f"\n===== {drug} =====")
    print(
        classification_report(
            y_test_xgb[drug],
            y_pred_xgb[drug]
        )
    )


===== ARTresistant =====
              precision    recall  f1-score   support

           0       0.94      0.98      0.96      4071
           1       0.87      0.67      0.76       811

    accuracy                           0.93      4882
   macro avg       0.90      0.83      0.86      4882
weighted avg       0.93      0.93      0.93      4882


===== CQresistant =====
              precision    recall  f1-score   support

           0       0.68      0.93      0.78      2399
           1       0.89      0.57      0.70      2483

    accuracy                           0.75      4882
   macro avg       0.79      0.75      0.74      4882
weighted avg       0.79      0.75      0.74      4882


===== MQresistant =====
              precision    recall  f1-score   support

           0       0.97      1.00      0.99      4756
           1       0.00      0.00      0.00       126

    accuracy                           0.97      4882
   macro avg       0.49      0.50      0.49      488

/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

In [91]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score
)

print(classification_report(y_test, y_pred))

print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_test, y_pr))

              precision    recall  f1-score   support

           0       0.87      0.67      0.76       811
           1       0.89      0.57      0.70      2483
           2       0.00      0.00      0.00       126
           3       0.00      0.00      0.00       128
           4       0.95      0.99      0.97      4379
           5       0.84      0.95      0.89      3846

   micro avg       0.90      0.84      0.87     11773
   macro avg       0.59      0.53      0.55     11773
weighted avg       0.88      0.84      0.85     11773
 samples avg       0.83      0.81      0.81     11773

Confusion matrix:


/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/ndono_work/miniforge3/envs/jp/lib/python3.14/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capita

ValueError: multilabel-indicator is not supported

## XGboost with weights 

In [92]:
for drug in y_train.columns:
    n0 = (y_train[drug] == 0).sum()
    n1 = (y_train[drug] == 1).sum()

    print(
        f"{drug}: "
        f"0={n0}, "
        f"1={n1}, "
        f"scale_pos_weight={n0/n1:.2f}"
    )

ARTresistant: 0=16433, 1=3094, scale_pos_weight=5.31
CQresistant: 0=9578, 1=9949, scale_pos_weight=0.96
MQresistant: 0=18995, 1=532, scale_pos_weight=35.70
PPQresistant: 0=19167, 1=360, scale_pos_weight=53.24
PYRresistant: 0=1942, 1=17585, scale_pos_weight=0.11
SDXresistant: 0=4103, 1=15424, scale_pos_weight=0.27


In [93]:
xgb_models = {}
y_pred_xgb = {}

for drug in y_train.columns:

    n0 = (y_train[drug] == 0).sum()
    n1 = (y_train[drug] == 1).sum()

    weight = n0 / n1

    model = XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=weight,
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train_xgb, y_train[drug])

    xgb_models[drug] = model
    y_pred_xgb[drug] = model.predict(X_test_xgb)

In [94]:
from sklearn.metrics import classification_report

for drug in y.columns:
    print(f"\n===== {drug} =====")
    print(
        classification_report(
            y_test_xgb[drug],
            y_pred_xgb[drug]
        )
    )


===== ARTresistant =====
              precision    recall  f1-score   support

           0       0.97      0.89      0.93      4071
           1       0.62      0.88      0.73       811

    accuracy                           0.89      4882
   macro avg       0.80      0.89      0.83      4882
weighted avg       0.92      0.89      0.90      4882


===== CQresistant =====
              precision    recall  f1-score   support

           0       0.68      0.93      0.78      2399
           1       0.89      0.57      0.70      2483

    accuracy                           0.75      4882
   macro avg       0.79      0.75      0.74      4882
weighted avg       0.79      0.75      0.74      4882


===== MQresistant =====
              precision    recall  f1-score   support

           0       0.99      0.90      0.94      4756
           1       0.17      0.81      0.28       126

    accuracy                           0.89      4882
   macro avg       0.58      0.85      0.61      488

### probabilities

In [95]:
y_prob = model.predict_proba(X_test_xgb)[:, 1]

In [96]:
import numpy as np
from sklearn.metrics import f1_score

thresholds = np.arange(0.1, 0.91, 0.05)

for threshold in thresholds:

    pred = (y_prob >= threshold).astype(int)

    f1 = f1_score(
        y_test_xgb["MQresistant"],
        pred
    )

    print(
        f"Threshold: {threshold:.2f} | F1: {f1:.3f}"
    )

Threshold: 0.10 | F1: 0.050
Threshold: 0.15 | F1: 0.057
Threshold: 0.20 | F1: 0.057
Threshold: 0.25 | F1: 0.057
Threshold: 0.30 | F1: 0.058
Threshold: 0.35 | F1: 0.058
Threshold: 0.40 | F1: 0.095
Threshold: 0.45 | F1: 0.101
Threshold: 0.50 | F1: 0.101
Threshold: 0.55 | F1: 0.101
Threshold: 0.60 | F1: 0.102
Threshold: 0.65 | F1: 0.102
Threshold: 0.70 | F1: 0.102
Threshold: 0.75 | F1: 0.102
Threshold: 0.80 | F1: 0.139
Threshold: 0.85 | F1: 0.143
Threshold: 0.90 | F1: 0.169


In [98]:
y_prob_ppq = model.predict_proba(X_test_xgb)[:, 1]
for threshold in thresholds:

    pred = (y_prob_ppq >= threshold).astype(int)

    f1 = f1_score(
        y_test_xgb["PPQresistant"],
        pred
    )

    print(
        f"Threshold: {threshold:.2f} | F1: {f1:.3f}"
    )

Threshold: 0.10 | F1: 0.051
Threshold: 0.15 | F1: 0.058
Threshold: 0.20 | F1: 0.058
Threshold: 0.25 | F1: 0.058
Threshold: 0.30 | F1: 0.058
Threshold: 0.35 | F1: 0.059
Threshold: 0.40 | F1: 0.097
Threshold: 0.45 | F1: 0.103
Threshold: 0.50 | F1: 0.103
Threshold: 0.55 | F1: 0.103
Threshold: 0.60 | F1: 0.104
Threshold: 0.65 | F1: 0.104
Threshold: 0.70 | F1: 0.104
Threshold: 0.75 | F1: 0.104
Threshold: 0.80 | F1: 0.145
Threshold: 0.85 | F1: 0.149
Threshold: 0.90 | F1: 0.181


In [105]:
ppq_model = xgb_models["PPQresistant"]

In [106]:
ppq_prob = ppq_model.predict_proba(X_test_xgb)[:, 1]

In [107]:
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.1, 0.95, 0.05)

for threshold in thresholds:

    ppq_pred = (ppq_prob >= threshold).astype(int)

    f1 = f1_score(
        y_test_xgb["PPQresistant"],
        ppq_pred,
        zero_division=0
    )

    print(f"Threshold: {threshold:.2f} | F1: {f1:.3f}")

Threshold: 0.10 | F1: 0.119
Threshold: 0.15 | F1: 0.119
Threshold: 0.20 | F1: 0.119
Threshold: 0.25 | F1: 0.121
Threshold: 0.30 | F1: 0.121
Threshold: 0.35 | F1: 0.128
Threshold: 0.40 | F1: 0.201
Threshold: 0.45 | F1: 0.205
Threshold: 0.50 | F1: 0.205
Threshold: 0.55 | F1: 0.205
Threshold: 0.60 | F1: 0.237
Threshold: 0.65 | F1: 0.237
Threshold: 0.70 | F1: 0.237
Threshold: 0.75 | F1: 0.237
Threshold: 0.80 | F1: 0.250
Threshold: 0.85 | F1: 0.250
Threshold: 0.90 | F1: 0.000


In [108]:
print(
    classification_report(
        y_test_xgb["PPQresistant"],
        (ppq_prob >= 0.5).astype(int),
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0       1.00      0.82      0.90      4754
           1       0.12      0.89      0.20       128

    accuracy                           0.82      4882
   macro avg       0.56      0.85      0.55      4882
weighted avg       0.97      0.82      0.88      4882



#### MQ


In [99]:
mq_model = xgb_models["MQresistant"]

In [100]:
mq_prob = mq_model.predict_proba(X_test_xgb)[:, 1]

In [101]:
from sklearn.metrics import f1_score
import numpy as np

thresholds = np.arange(0.1, 0.95, 0.05)

for threshold in thresholds:

    mq_pred = (mq_prob >= threshold).astype(int)

    f1 = f1_score(
        y_test_xgb["MQresistant"],
        mq_pred,
        zero_division=0
    )

    print(f"Threshold: {threshold:.2f} | F1: {f1:.3f}")

Threshold: 0.10 | F1: 0.106
Threshold: 0.15 | F1: 0.113
Threshold: 0.20 | F1: 0.113
Threshold: 0.25 | F1: 0.113
Threshold: 0.30 | F1: 0.113
Threshold: 0.35 | F1: 0.162
Threshold: 0.40 | F1: 0.161
Threshold: 0.45 | F1: 0.259
Threshold: 0.50 | F1: 0.281
Threshold: 0.55 | F1: 0.290
Threshold: 0.60 | F1: 0.290
Threshold: 0.65 | F1: 0.290
Threshold: 0.70 | F1: 0.324
Threshold: 0.75 | F1: 0.360
Threshold: 0.80 | F1: 0.423
Threshold: 0.85 | F1: 0.423
Threshold: 0.90 | F1: 0.423


In [102]:
print(
    classification_report(
        y_test_xgb["MQresistant"],
        (mq_prob >= 0.5).astype(int),
        zero_division=0
    )
)

              precision    recall  f1-score   support

           0       0.99      0.90      0.94      4756
           1       0.17      0.81      0.28       126

    accuracy                           0.89      4882
   macro avg       0.58      0.85      0.61      4882
weighted avg       0.97      0.89      0.93      4882



In [103]:
from sklearn.metrics import average_precision_score

mq_pr_auc = average_precision_score(
    y_test_xgb["MQresistant"],
    mq_prob
)

print("MQ PR-AUC:", mq_pr_auc)

MQ PR-AUC: 0.25742035928231877


In [104]:
from sklearn.metrics import average_precision_score, f1_score

mq_train_prob = xgb_models["MQresistant"].predict_proba(
    X_train_xgb
)[:, 1]

mq_test_prob = xgb_models["MQresistant"].predict_proba(
    X_test_xgb
)[:, 1]

print("Train PR-AUC:",
      average_precision_score(y_train_xgb["MQresistant"], mq_train_prob))

print("Test PR-AUC:",
      average_precision_score(y_test_xgb["MQresistant"], mq_test_prob))

Train PR-AUC: 0.2481986925850983
Test PR-AUC: 0.25742035928231877


## Haplotypes associated with drug resistance

In [ ]:
# columns for mutations in Sulfadoxine-Pyrimethamine (treatment) and Sulfadoxine-Pyrimethamine (IPTp)
dhfr_cols = [
    "dhfr_51[N]",
    "dhfr_59[C]",
    "dhfr_108[S]",
    "dhfr_164[I]"
]

dhps_cols = [
    "dhps_437[G]",
    "dhps_540[K]",
    "dhps_581[A]",
    "dhps_613[A]"
]

combo_cols = dhfr_cols + dhps_cols

In [ ]:
# dealing with SP-IPTp (sextuple mutant)
def classify_sp(row):

    cols = [
        "dhfr_51[N]",
        "dhfr_59[C]",
        "dhfr_108[S]",
        "dhfr_164[I]",
        "dhps_437[G]",
        "dhps_540[K]",
        "dhps_581[A]",
        "dhps_613[A]"
    ]

    if any(is_ambiguous(row[c]) for c in cols):
        return "Rare"

    if (
        row["dhfr_51[N]"] == "I"
        and row["dhfr_59[C]"] == "R"
        and row["dhfr_108[S]"] == "N"
        and row["dhps_437[G]"] == "G"
        and row["dhps_540[K]"] == "E"
        and (
            row["dhfr_164[I]"] == "L"
            or row["dhps_581[A]"] == "G"
            or row["dhps_613[A]"] in {"S", "T"}
        )
    ):
        return "Resistant"

    return "Sensitive"

In [ ]:
# helper function
def is_ambiguous(value):

    if pd.isna(value):
        return True

    value = str(value).strip()

    if value in {"-", "*", "!"}:
        return True

    return value.islower()

In [ ]:
# dealing with Pyrimethamine (triple mutant)
def classify_pyr(row):

    cols = [
        "dhfr_51[N]",
        "dhfr_59[C]",
        "dhfr_108[S]"
    ]

    if any(is_ambiguous(row[c]) for c in cols):
        return "Rare"

    if (
        row["dhfr_51[N]"] == "I"
        and row["dhfr_59[C]"] == "R"
        and row["dhfr_108[S]"] == "N"
    ):
        return "Resistant"

    return "Sensitive"

In [ ]:
# pyresistant is for triplet Pyrimethamine
encoded["PYRresistant"] = genotype.apply(classify_pyr, axis=1)
# Spresistant if for sextuple mutant
encoded["SPresistant"] = genotype.apply(classify_sp, axis=1)

In [ ]:
encoded.head()

,Sample,crt_76[K],dhfr_108[S],dhps_437[G],dhfr_51[N]_assoc,dhfr_59[C]_assoc,dhfr_164[I]_assoc,dhps_540[K]_assoc,dhps_581[A]_assoc,dhps_613[A]_assoc,...,mdr1_184[Y],mdr1_1034[S],mdr1_1042[N],mdr1_1226[F],mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T],PYRresistant,SPresistant
0,FP0008-C,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,Sensitive,Sensitive
1,FP0009-C,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,sensitive,sensitive,sensitive,Rare,sensitive,sensitive,sensitive,Resistant,Sensitive
2,FP0010-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,Resistant,Sensitive
3,FP0011-CW,Resistant,Resistant,Resistant,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Rare,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,Resistant,Sensitive
4,FP0012-CW,Resistant,Resistant,Sensitive,Resistant,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,sensitive,Resistant,Sensitive


In [ ]:
def encode(value, column):

    # Missing
    if pd.isna(value) or str(value).strip() == "":
        return "Missing"

    value = str(value).strip()

    # ---------- Special: copy number ----------
    # ----- Special: copy number ----------
    if column in ["mdr1_dup_call", "pm2_dup_call"]:
        if int(value) == 0:
            return "Sensitive"
        elif int(value) == 1:
            return "Resistant"
        elif int(value) in {"-", "*", "!"}:
            return "missing"
        else:
            return "rare"
    

    # ---------- Special: crt haplotype ----------
    if column == "crt_72-76[CVMNK]":

        haps = [x.strip() for x in value.split(",")]

        if any(h in {"CVIET", "SVMNT"} for h in haps):
            return "Resistant"

        if all(h == "CVMNK" for h in haps):
            return "Sensitive"

        return "Rare"

    # ---------- Special: kelch13 ----------
    if column == "kelch13_349-726_ns_changes":

        # Missing or unresolved
        if pd.isna(value):
            return "missing"

        value = str(value).strip()

        if value in {"-", "*", "!"}:
            return "missing"

        # Two haplotypes
        if "," in value:
            haps = [x.strip() for x in value.split(",")]

            # Same mutation on both haplotypes
            if len(set(haps)) == 1:
                value = haps[0]
            else:
                return "Rare"

        # Lowercase = heterozygous
        if value.islower():
            return "Rare"

        value = value.upper()

        # Wild type (if present in your data)
        if value in {"WT", "578S"}:
            return "Sensitive"

        # WHO resistance mutation
        if value in WHO_K13:
            return "Resistant"

        # Any other homozygous mutation
        return "Sensitive"
    
    

    # ---------- Multiple alleles ----------
    alleles = [a.strip() for a in value.split(",")]
    if len(alleles) > 1:

        if column in SPECIAL:
            if any(a in SPECIAL[column] for a in alleles):
                return "Resistant"
            return "Rare"

        # General columns
        if ref_aa[column] in alleles:
            return "Sensitive"

        return "Rare"

    # ---------- Single allele ----------

    allele = alleles[0]

    if column in SPECIAL:
        if allele in SPECIAL[column]:
            return "rare"

    if allele == ref_aa[column]:
        return "Sensitive"

    return "missing"

### Handling artesunate_mefloquine

In [ ]:
def classify_artesunate_mefloquine(row):

    art = row["kelch13_349-726_ns_changes"]
    mq = row["mdr1_dup_call"]

    # Missing
    if art == "Missing" or mq == "Missing":
        return "Missing"

    # Both resistant
    if art == "Resistant" and mq == "Resistant":
        return "Resistant"

    # At least one sensitive
    if art == "Sensitive" or mq == "Sensitive":
        return "Sensitive"

    # Rare/unknown combinations
    return "Rare"

In [ ]:
encoded["ASMQresistant"] = encoded.apply(
    classify_artesunate_mefloquine,
    axis=1
)

KeyError: 'kelch13_349-726_ns_changes'

### Handling piperaquine

In [ ]:
encoded.columns

Index(['Sample', 'crt_72[C]', 'crt_74[M]', 'crt_75[N]', 'crt_76[K]',
       'crt_72-76[CVMNK]', 'crt_93[T]', 'crt_97[H]', 'crt_218[I]',
       'crt_220[A]', 'crt_271[Q]', 'crt_326[N]', 'crt_333[T]', 'crt_353[G]',
       'crt_356[I]', 'crt_371[R]', 'dhfr_16[N]', 'dhfr_51[N]', 'dhfr_59[C]',
       'dhfr_108[S]', 'dhfr_164[I]', 'dhfr_306[S]', 'dhps_436[S]',
       'dhps_437[G]', 'dhps_540[K]', 'dhps_581[A]', 'dhps_613[A]',
       'exo_415[E]', 'mdr1_86[N]', 'mdr1_184[Y]', 'mdr1_1034[S]',
       'mdr1_1042[N]', 'mdr1_1226[F]', 'mdr1_1246[D]', 'arps10_127-128[VD]',
       'fd_193[D]', 'mdr2_484[T]', 'kelch13_349-726_ns_changes',
       'mdr1_dup_call', 'pm2_dup_call', 'PYRresistant', 'SPresistant',
       'ASMQresistant'],
      dtype='str')

In [ ]:
def classify_artesunate_mefloquine(row):

    art = row["kelch13_349-726_ns_changes"]
    mq = row["mdr1_dup_call"]

    # Missing
    if art == "Missing" or mq == "Missing":
        return "Missing"

    # Both resistant
    if art == "Resistant" and mq == "Resistant":
        return "Resistant"

    # At least one sensitive
    if art == "Sensitive" or mq == "Sensitive":
        return "Sensitive"

    # Rare/unknown combinations
    return "Rare"

In [ ]:
encoded

,Sample,crt_72[C],crt_74[M],crt_75[N],crt_76[K],crt_72-76[CVMNK],crt_93[T],crt_97[H],crt_218[I],crt_220[A],...,mdr1_1246[D],arps10_127-128[VD],fd_193[D],mdr2_484[T],kelch13_349-726_ns_changes,mdr1_dup_call,pm2_dup_call,PYRresistant,SPresistant,ASMQresistant
0,FP0008-C,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Sensitive,Sensitive,Missing
1,FP0009-C,Sensitive,missing,missing,missing,Resistant,Sensitive,Sensitive,Sensitive,missing,...,missing,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
2,FP0010-CW,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
3,FP0011-CW,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
4,FP0012-CW,Sensitive,missing,missing,missing,Resistant,Sensitive,Sensitive,Sensitive,missing,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,Sensitive,Resistant,Sensitive,Missing
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24404,SPT92049,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,missing,Sensitive,Sensitive,Sensitive,missing,rare,rare,Resistant,Rare,Rare
24405,SPT92054,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,Missing,Sensitive,rare,Resistant,Sensitive,Missing
24406,SPT92057,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,Sensitive,...,Sensitive,Sensitive,Sensitive,Sensitive,missing,rare,rare,Resistant,Sensitive,Rare
24407,SPT94772,Sensitive,missing,missing,missing,Resistant,Sensitive,Sensitive,Sensitive,missing,...,missing,Sensitive,Sensitive,missing,missing,rare,rare,Resistant,Sensitive,Rare


## Filtering African countries only

,sample_id,year,qc_pass,study_id,region,country,country_id,site,site_id,ARTresistant,CQresistant,MQresistant,PPQresistant,PYRresistant,SDXresistant
0,FP0008-C,2014,True,1147,NaN,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,sensitive,sensitive,undetermined,undetermined
1,FP0009-C,2014,True,1147,NaN,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,sensitive,sensitive,resistant,sensitive
2,FP0010-CW,2014,True,1147,NaN,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,resistant
3,FP0011-CW,2014,True,1147,NaN,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,undetermined
4,FP0012-CW,2014,True,1147,NaN,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,undetermined,undetermined,resistant,sensitive


In [ ]:
metadata['ARTresistant'].unique()

<StringArray>
['sensitive', nan, 'undetermined', 'resistant']
Length: 4, dtype: str

In [ ]:
metadata['ARTresistant'].value_counts()

ARTresistant
sensitive       17147
resistant        3905
undetermined     3357
Name: count, dtype: int64

In [ ]:
metadata['ARTresistant'].isna().sum()

np.int64(8586)

In [ ]:
metadata['country'].unique()

<StringArray>
[                      'Mauritania',                           'Gambia',
                           'Guinea',                            'Kenya',
                         'Thailand',                         'Tanzania',
                            'Ghana',                         'Cambodia',
                        'Indonesia',                     'Burkina Faso',
                             'Mali',                 'Papua New Guinea',
                             'Peru',                       'Bangladesh',
                           'Malawi',                          'Vietnam',
                         'Colombia',                        'Venezuela',
                           'Uganda',                          'Myanmar',
                             'Laos', 'Democratic Republic of the Congo',
                          'Nigeria',                       'Madagascar',
                         'Cameroon',                    'Côte d'Ivoire',
                         'Ethiopia', 

<class 'pandas.DataFrame'>
Index: 20951 entries, 0 to 32994
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   sample_id     20951 non-null  str    
 1   year          20951 non-null  int64  
 2   qc_pass       20951 non-null  bool   
 3   study_id      20951 non-null  int64  
 4   region        0 non-null      float64
 5   country       20951 non-null  str    
 6   country_id    20951 non-null  str    
 7   site          20951 non-null  str    
 8   site_id       20951 non-null  str    
 9   ARTresistant  14508 non-null  str    
 10  CQresistant   14508 non-null  str    
 11  MQresistant   14508 non-null  str    
 12  PPQresistant  14508 non-null  str    
 13  PYRresistant  14508 non-null  str    
 14  SDXresistant  14508 non-null  str    
dtypes: bool(1), float64(1), int64(2), str(11)
memory usage: 2.4 MB


In [ ]:
africa_df.columns

Index(['sample_id', 'year', 'qc_pass', 'study_id', 'country', 'country_id',
       'site', 'site_id', 'ARTresistant', 'CQresistant', 'MQresistant',
       'PPQresistant', 'PYRresistant', 'SDXresistant'],
      dtype='str')

In [ ]:
print(len(african_countries_geno))

20951


In [ ]:
african_countries_geno.head()

,Sample,crt_72[C],crt_74[M],crt_75[N],crt_76[K],crt_72-76[CVMNK],crt_93[T],crt_97[H],crt_218[I],crt_220[A],...,country,country_id,site,site_id,ARTresistant,CQresistant,MQresistant,PPQresistant,PYRresistant_y,SDXresistant
0,FP0008-C,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,sensitive,sensitive,undetermined,undetermined
1,FP0009-C,Sensitive,missing,missing,missing,Resistant,Sensitive,Sensitive,Sensitive,missing,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,sensitive,sensitive,resistant,sensitive
2,FP0010-CW,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,resistant
3,FP0011-CW,Sensitive,Sensitive,Sensitive,Sensitive,Resistant,Sensitive,Sensitive,Sensitive,Sensitive,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,undetermined,undetermined,undetermined,resistant,undetermined
4,FP0012-CW,Sensitive,missing,missing,missing,Resistant,Sensitive,Sensitive,Sensitive,missing,...,Mauritania,MR,Hodh el Gharbi,Hodh el Gharbi,sensitive,resistant,undetermined,undetermined,resistant,sensitive


In [ ]:
african_countries_geno.info()

<class 'pandas.DataFrame'>
RangeIndex: 20951 entries, 0 to 20950
Data columns (total 57 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   Sample                      14508 non-null  str  
 1   crt_72[C]                   14508 non-null  str  
 2   crt_74[M]                   14508 non-null  str  
 3   crt_75[N]                   14508 non-null  str  
 4   crt_76[K]                   14508 non-null  str  
 5   crt_72-76[CVMNK]            14508 non-null  str  
 6   crt_93[T]                   14508 non-null  str  
 7   crt_97[H]                   14508 non-null  str  
 8   crt_218[I]                  14508 non-null  str  
 9   crt_220[A]                  14508 non-null  str  
 10  crt_271[Q]                  14508 non-null  str  
 11  crt_326[N]                  14508 non-null  str  
 12  crt_333[T]                  14508 non-null  str  
 13  crt_353[G]                  14508 non-null  str  
 14  crt_356[I]       

# Merging with East African Countries

In [ ]:
# loading east africa data 
east_genotype = pd.read_csv('../processed/east-country_year2010-2019.csv')

In [ ]:
east_genotype.columns

Index(['Unnamed: 0', 'sample_id', 'year', 'qc_pass', 'study_id', 'region',
       'country', 'country_id', 'site', 'site_id', 'ARTresistant',
       'CQresistant', 'MQresistant', 'PPQresistant', 'PYRresistant',
       'SDXresistant'],
      dtype='str')

In [ ]:
# saving East africa countries encoded genotype data
east_african_geno.to_csv('../processed/East_africa_geno.csv')

# Merging with West African Countries